# Drug Response Prediction with Measured Profiles

Stratified T1-T4 Pearson for SMILES modalities (SciPlex within cell line;
McFarland within tissue / drug), plus supplementary tables and manuscript stats.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr,ttest_ind, spearmanr
from sklearn.metrics import mean_squared_error
import numpy as np


In [ ]:
# Get the project root directory
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "config.py").exists():
    project_root = project_root.parents[1]
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "scripts"))
sys.path.insert(0, str(project_root / "scripts" / "04_predict_drug_response"))
from notebook_imports import *



In [ ]:
# Set project directories using configuration
home_dir = str(config.PROJECT_ROOT)
data_dir = str(config.DATA_DIR)
resources_dir = str(config.RESOURCES_DIR)
results_dir = str(config.RESULTS_04_DIR)
figures_dir = str(config.FIGURES_04_DIR)


In [ ]:
# Color palettes
pre_post_lfc_palette = {
    'Grey': '#808080',       # Standard grey
    'Red': '#FF0000',        # Bright red
    'Dark Red': '#8B0000'    # Dark red
}
pre_post_lfc_palette = list(pre_post_lfc_palette.values())

TASK_MODEL_ORDER = ['Pre', 'Post', 'LFC']
SMILES_MODEL_ORDER = ['Pre', 'Pre+SMILES', 'Post', 'Post+SMILES', 'LFC', 'LFC+SMILES']
ONLY_SMILES_MODEL_ORDER = ['Pre+SMILES', 'Post+SMILES', 'LFC+SMILES']

task_model_palette = {
    'Pre': '#808080',
    'Pre+SMILES': '#808080',
    'Post': '#FF0000',
    'Post+SMILES': '#FF0000',
    'LFC': '#8B0000',
    'LFC+SMILES': '#8B0000',
}
sens_res_palette = {
    'Resistant': 'darkgrey',  # plum
    'Sensitive': '#FF0000'   # olive
}


## Stratified T1-T4 Pearson (within line / drug)

Three panels matching the predicted-profile stratification:

1. SciPlex3 within cell line
2. McFarland within tissue
3. McFarland within drug

Uses measured-task OOF predictions. **T1 uses the same predefined 5-fold splits**
as predicted-profile CV (`cv_scheme=predefined_fold` from `*_t1_5fold_predictions.csv`).


In [ ]:
import importlib
import create_figures_tasks
importlib.reload(create_figures_tasks)
from create_figures_tasks import (
    load_stratified_task_panel_scores,
    plot_tasks_stratified_three_panel,
)

# SMILES modalities (same as measured-task SMILES boxplots above).
_strat_models = list(ONLY_SMILES_MODEL_ORDER)

# T1 = predefined 5-fold; T2-T4 = exhaustive LOO.
# align_common_groups=True → every boxplot in a panel shares the same groups.
strat_panels = load_stratified_task_panel_scores(
    include_two_stage=False,
    models=_strat_models,
    t1_cv_scheme='predefined_fold',
    align_common_groups=True,
)

for key, frame in strat_panels.items():
    if frame.empty:
        print(f'{key}: no scored rows yet')
        continue
    for task in ['T1', 'T2', 'T3', 'T4']:
        t = frame[frame['task'] == task]
        if t.empty:
            continue
        groups_by_model = {
            m: sorted(g['group'].astype(str).unique())
            for m, g in t.groupby(t['model'].astype(str))
        }
        n_per_model = {m: len(gs) for m, gs in groups_by_model.items()}
        shared = set.intersection(*(set(gs) for gs in groups_by_model.values())) if groups_by_model else set()
        print(
            f'{key} {task}: n_per_model={n_per_model}, '
            f'same_groups={all(len(gs) == len(shared) for gs in groups_by_model.values())}, '
            f'n_common={len(shared)}'
        )

fig = plot_tasks_stratified_three_panel(
    strat_panels,
    model_order=_strat_models,
    palette=task_model_palette,
    outfile=os.path.join(
        figures_dir, 'measured_tasks_stratified_line_drug_pearson.pdf'
    ),
    ylim=(-1.0, 1.0),
)
plt.show()



## Stratified T1-T4 Pearson without SMILES

Same three-panel layout (SciPlex within line | McFarland within tissue | within drug), using expression-only modalities `Pre` / `Post` / `LFC` from `OUTPUT_TAG=nosmiles`.


In [ ]:
import importlib
import os
from pathlib import Path

import create_figures_tasks

importlib.reload(create_figures_tasks)
from create_figures_tasks import (
    load_stratified_task_panel_scores,
    plot_tasks_stratified_three_panel,
)

_nosmiles_pred = Path(results_dir) / 'sciplex_nosmiles_measured_tasks_predictions.csv'
_nosmiles_mcf = Path(results_dir) / 'mcfarland_nosmiles_measured_tasks_predictions.csv'
if not (_nosmiles_pred.exists() or _nosmiles_mcf.exists()):
    print(
        'No-SMILES stratified outputs missing. Submit:\n'
        '  OUTPUT_TAG=nosmiles MEASURED_MODELS=Pre,Post,LFC \\n'
        '    bash scripts/04_predict_drug_response/submit_measured_task_cv.sh'
    )
else:
    _strat_models_nosmiles = list(TASK_MODEL_ORDER)  # Pre, Post, LFC
    strat_panels_nosmiles = load_stratified_task_panel_scores(
        include_two_stage=False,
        models=_strat_models_nosmiles,
        t1_cv_scheme='predefined_fold',
        align_common_groups=True,
        output_tag='nosmiles',
    )

    for key, frame in strat_panels_nosmiles.items():
        if frame.empty:
            print(f'{key}: no scored rows yet')
            continue
        for task in ['T1', 'T2', 'T3', 'T4']:
            t = frame[frame['task'] == task]
            if t.empty:
                continue
            groups_by_model = {
                m: sorted(g['group'].astype(str).unique())
                for m, g in t.groupby(t['model'].astype(str))
            }
            n_per_model = {m: len(gs) for m, gs in groups_by_model.items()}
            shared = (
                set.intersection(*(set(gs) for gs in groups_by_model.values()))
                if groups_by_model
                else set()
            )
            print(
                f'{key} {task}: n_per_model={n_per_model}, '
                f'same_groups={all(len(gs) == len(shared) for gs in groups_by_model.values())}, '
                f'n_common={len(shared)}'
            )

    fig = plot_tasks_stratified_three_panel(
        strat_panels_nosmiles,
        model_order=_strat_models_nosmiles,
        palette=task_model_palette,
        outfile=os.path.join(
            figures_dir, 'measured_tasks_stratified_line_drug_pearson_nosmiles.pdf'
        ),
        ylim=(-1.0, 1.0),
    )
    plt.show()



## SMILES measured-task stratified summary (Supplementary Data)

Mean ± s.d. Pearson / Spearman / RMSE across the **same** groups for Pre+SMILES / Post+SMILES / LFC+SMILES, plus paired Δ [95% CI] and $p$ vs Pre+Morgan for all three metrics.

Panels match the stratified figure: SciPlex within cell line, McFarland within tissue, McFarland within drug.
Each panel writes Pearson / Spearman / RMSE tables that share one page in the supplement.
T1 = predefined 5-fold; T2-T4 = exhaustive LOO.

Writes CSV under `results/04_predict_drug_response/` and LaTeX under `figures/04_predict_drug_response/`
(`measured_tasks_smiles_stratified_summary_*.tex`), included in `Supplementary_Data.tex` section 3.


In [ ]:
from create_figures_tasks import (
    build_measured_tasks_stratified_summary,
    write_measured_tasks_stratified_summary,
)

_smiles_models = list(ONLY_SMILES_MODEL_ORDER)
# Always rebuild SMILES stratified panels (do not reuse nosmiles strat_panels*).
_smiles_summary = build_measured_tasks_stratified_summary(
    panel_scores=None,
    models=_smiles_models,
    t1_cv_scheme='predefined_fold',
)
display(
    _smiles_summary[
        [
            'panel',
            'dataset',
            'task',
            'modality',
            'n_groups',
            'pearson_mean',
            'pearson_sd',
            'spearman_mean',
            'spearman_sd',
            'rmse_mean',
            'rmse_sd',
            'pearson_vs_pre_smiles_delta',
            'pearson_vs_pre_smiles_sig',
        ]
    ].round(3)
)

_csv_path, _tex_paths = write_measured_tasks_stratified_summary(
    _smiles_summary,
    models=_smiles_models,
    wrap_landscape=False,
)
print(f'Wrote {_csv_path} ({len(_smiles_summary)} rows)')
for p in _tex_paths:
    print(f'Wrote {p}')
print(
    'Included in figures/submission_figures/Supplementary_Data.tex section 3 via '
    'measured_tasks_smiles_stratified_summary_{sciplex_within_line,'
    'mcfarland_within_tissue,mcfarland_within_drug}.tex'
)



## Manuscript statistics (Figure 1c,d claims)

Mean ± SD and paired modality / task contrasts for **SMILES** measured-profile T1–T4
(`Pre+SMILES` / `Post+SMILES` / `LFC+SMILES`).

Also regenerates Supplementary Data section 3 stratified tables
(`measured_tasks_smiles_stratified_summary_*.tex`).
Pooled / LFC-vs-Pre tables are written for manuscript stats but are **not**
included in the current supplement.

Uses figure numbering (**T2** = unseen tissue/cell line, **T3** = unseen drug).


In [ ]:
# Manuscript-facing stats for Figure 1c,d claims (measured-profile T1-T4).
# Uses SMILES modalities only (Pre+SMILES / Post+SMILES / LFC+SMILES).
# Task codes match task_cv / boxplot axis: T2=unseen context, T3=unseen drug.

from itertools import combinations
from scipy.stats import ttest_rel, ttest_ind

import importlib
import create_figures_tasks

importlib.reload(create_figures_tasks)
from create_figures_tasks import (
    build_measured_tasks_lfc_vs_pre_tests,
    build_measured_tasks_pooled_summary,
    build_measured_tasks_stratified_summary,
    load_grouped_task_scores,
    write_measured_tasks_lfc_vs_pre_tests,
    write_measured_tasks_pooled_summary,
    write_measured_tasks_stratified_summary,
)


def _mean_sd(vals: pd.Series) -> str:
    vals = pd.to_numeric(vals, errors='coerce').dropna()
    if vals.empty:
        return '--'
    if len(vals) == 1:
        return f'{vals.iloc[0]:.3f} (n=1)'
    return f'{vals.mean():.3f} +/- {vals.std(ddof=1):.3f} (n={len(vals)})'


def _paired_or_welch(a: pd.Series, b: pd.Series, index) -> tuple[float, float, int, str]:
    """Return mean(b-a), sd, n, test label."""
    wide = pd.DataFrame({'a': a, 'b': b}).dropna()
    if len(wide) >= 2 and index is not None:
        diff = wide['b'] - wide['a']
        _, p = ttest_rel(wide['b'], wide['a'])
        sd = float(diff.std(ddof=1)) if len(diff) > 1 else 0.0
        return float(diff.mean()), sd, len(diff), f'paired p={p:.3g}'
    aa = pd.to_numeric(a, errors='coerce').dropna()
    bb = pd.to_numeric(b, errors='coerce').dropna()
    if len(aa) < 2 or len(bb) < 2:
        return float('nan'), float('nan'), 0, 'insufficient n'
    _, p = ttest_ind(bb, aa, equal_var=False)
    return float(bb.mean() - aa.mean()), float('nan'), min(len(aa), len(bb)), f'Welch p={p:.3g}'


def print_manuscript_task_stats(df: pd.DataFrame, dataset: str, model_order=None) -> None:
    model_order = model_order or list(ONLY_SMILES_MODEL_ORDER)
    sub = df.query('dataset == @dataset').copy()
    sub['pearson'] = pd.to_numeric(sub['pearson'], errors='coerce').fillna(0.0)

    print('=' * 96)
    print(f'{dataset}: modality performance by task (SMILES)')
    print('=' * 96)
    for task in ('T1', 'T2', 'T3', 'T4'):
        tsub = sub[sub['task'] == task]
        if tsub.empty:
            continue
        print(f'\n{task}')
        for model in model_order:
            vals = tsub.loc[tsub['model'] == model, 'pearson']
            print(f'  {model:<12} {_mean_sd(vals)}')

        tmp = tsub[['group', 'model', 'pearson']].dropna()
        wide = tmp.groupby(['group', 'model'], as_index=False)['pearson'].mean()
        wide = wide.pivot(index='group', columns='model', values='pearson')
        print('  Modality deltas (paired on group):')
        for m1, m2 in combinations([m for m in model_order if m in wide.columns], 2):
            paired = wide[[m1, m2]].dropna()
            if len(paired) < 2:
                continue
            diff = paired[m2] - paired[m1]
            _, p = ttest_rel(paired[m2], paired[m1])
            sd = float(diff.std(ddof=1)) if len(diff) > 1 else 0.0
            print(
                f'    {m2:<12} - {m1:<12}: '
                f'{diff.mean():+.3f} +/- {sd:.3f} (n={len(diff)}, paired p={p:.3g})'
            )

    print('\n' + '=' * 96)
    print(f'{dataset}: task-to-task contrasts within modality (claims in text)')
    print('=' * 96)
    contrasts = [
        ('drug hold-out', 'T1', 'T3'),
        ('context hold-out', 'T1', 'T2'),
        ('context hold-out | unseen drug', 'T3', 'T4'),
        ('drug hold-out | unseen context', 'T2', 'T4'),
    ]
    for model in model_order:
        print(f'\n{model}')
        msub = sub[sub['model'] == model]
        for label, t_a, t_b in contrasts:
            a = msub[msub['task'] == t_a].set_index('group')['pearson']
            b = msub[msub['task'] == t_b].set_index('group')['pearson']
            shared = a.index.intersection(b.index)
            if len(shared) >= 2:
                mean_d, sd, n, note = _paired_or_welch(a.loc[shared], b.loc[shared], shared)
                sd_s = f'{sd:.3f}' if pd.notna(sd) else 'NA'
                print(
                    f'  {label:32} {t_a}->{t_b}: '
                    f'{_mean_sd(a)} -> {_mean_sd(b)}; '
                    f'delta={mean_d:+.3f} +/- {sd_s} ({note}, n={n})'
                )
            else:
                mean_d, sd, n, note = _paired_or_welch(a, b, None)
                print(
                    f'  {label:32} {t_a}->{t_b}: '
                    f'{_mean_sd(a)} -> {_mean_sd(b)}; '
                    f'delta={mean_d:+.3f} ({note})'
                )


_smiles_models = list(ONLY_SMILES_MODEL_ORDER)

# Prefer in-memory SMILES scores from the pooled figure cell; else reload.
if 'smiles_scores' in globals() and len(smiles_scores):
    _stats_df = smiles_scores.copy()
elif 'grouped_scores' in globals() and len(grouped_scores):
    _stats_df = grouped_scores[
        grouped_scores['model'].astype(str).isin(_smiles_models)
    ].copy()
else:
    _stats_df = load_grouped_task_scores(
        include_two_stage=False, t1_cv_scheme='predefined_fold'
    )
    _stats_df = _stats_df[
        _stats_df['model'].astype(str).isin(_smiles_models)
    ].copy()
_stats_df['pearson'] = pd.to_numeric(_stats_df['pearson'], errors='coerce').fillna(0.0)

print(
    'NOTE: SMILES modalities only (Pre+SMILES / Post+SMILES / LFC+SMILES). '
    'Figure / task_cv numbering is T2=unseen context, T3=unseen drug.'
)
print(
    'Loaded models:',
    sorted(_stats_df['model'].astype(str).unique()),
    '| n=',
    len(_stats_df),
)
print_manuscript_task_stats(_stats_df, 'SciPlex3', model_order=_smiles_models)
print_manuscript_task_stats(_stats_df, 'McFarland', model_order=_smiles_models)

# Compact claim-oriented summary for LFC+SMILES (main text modality).
print('\n' + '#' * 96)
print('Compact LFC+SMILES claim checks (mean +/- SD; delta = second - first)')
print('#' * 96)
for dataset, checks in (
    (
        'SciPlex3',
        [
            ('drugs held out', 'T1', 'T3'),
            ('unseen cell lines', 'T1', 'T2'),
            ('unseen cell lines | unseen drug', 'T3', 'T4'),
        ],
    ),
    (
        'McFarland',
        [
            ('tissue/context hold-out', 'T1', 'T2'),
            ('tissue/context | unseen drug', 'T3', 'T4'),
            ('drugs held out', 'T1', 'T3'),
        ],
    ),
):
    print(f'\n{dataset} (LFC+SMILES)')
    d = _stats_df.query('dataset == @dataset and model == "LFC+SMILES"')
    for label, t_a, t_b in checks:
        a = d.loc[d['task'] == t_a, 'pearson']
        b = d.loc[d['task'] == t_b, 'pearson']
        print(f'  {label}: {_mean_sd(a)} vs {_mean_sd(b)}  (delta={(b.mean()-a.mean()):+.3f})')

print('\nPost+SMILES vs Pre+SMILES / LFC+SMILES across tasks:')
for dataset in ('SciPlex3', 'McFarland'):
    d = _stats_df.query('dataset == @dataset')
    print(f'\n{dataset}')
    for task in ('T1', 'T2', 'T3', 'T4'):
        t = d[d['task'] == task]
        pre = t.loc[t['model'] == 'Pre+SMILES', 'pearson']
        post = t.loc[t['model'] == 'Post+SMILES', 'pearson']
        lfc = t.loc[t['model'] == 'LFC+SMILES', 'pearson']
        wide = (
            t[['group', 'model', 'pearson']]
            .groupby(['group', 'model'], as_index=False)['pearson'].mean()
            .pivot(index='group', columns='model', values='pearson')
        )
        parts = [
            f'{task}: Pre+SMILES {_mean_sd(pre)}; '
            f'Post+SMILES {_mean_sd(post)}; '
            f'LFC+SMILES {_mean_sd(lfc)}'
        ]
        if {'Pre+SMILES', 'Post+SMILES'}.issubset(wide.columns):
            p = wide[['Pre+SMILES', 'Post+SMILES']].dropna()
            if len(p) >= 2:
                _, pv = ttest_rel(p['Post+SMILES'], p['Pre+SMILES'])
                parts.append(
                    f'Post-Pre={(p["Post+SMILES"]-p["Pre+SMILES"]).mean():+.3f} (p={pv:.3g})'
                )
        if {'Post+SMILES', 'LFC+SMILES'}.issubset(wide.columns):
            p = wide[['Post+SMILES', 'LFC+SMILES']].dropna()
            if len(p) >= 2:
                _, pv = ttest_rel(p['LFC+SMILES'], p['Post+SMILES'])
                parts.append(
                    f'LFC-Post={(p["LFC+SMILES"]-p["Post+SMILES"]).mean():+.3f} (p={pv:.3g})'
                )
        print('  ' + '; '.join(parts))


# Paired LFC+SMILES vs Pre+SMILES for every dataset x task (+ stratified panels)
_lfc_vs_pre = build_measured_tasks_lfc_vs_pre_tests(
    _stats_df,
    lfc_model='LFC+SMILES',
    pre_model='Pre+SMILES',
    t1_cv_scheme='predefined_fold',
    include_stratified=True,
)
print('\n' + '#' * 96)
print('LFC+SMILES vs Pre+SMILES (paired on group; every dataset x task)')
print('#' * 96)
for _, row in _lfc_vs_pre.iterrows():
    print(
        f"  {row['panel']:<18} {row['dataset']:<10} {row['task']}: "
        f"Pre={row['pre_pearson_mean']:.3f}, LFC={row['lfc_pearson_mean']:.3f}, "
        f"delta={row['delta_lfc_minus_pre']:+.3f}, p={row['p_value']:.3g}, "
        f"{row['significance']} (n={int(row['n_groups'])})"
    )
_lfc_csv, _lfc_tex = write_measured_tasks_lfc_vs_pre_tests(_lfc_vs_pre)
print(f'Wrote {_lfc_csv}')
print(f'Wrote {_lfc_tex}')
display(_lfc_vs_pre.round(3))

# --- Tables: pooled/LFC optional; stratified feeds Supplementary_Data.tex section 3 ---
_pooled_summary = build_measured_tasks_pooled_summary(
    _stats_df,
    models=_smiles_models,
    t1_cv_scheme='predefined_fold',
)
_pooled_csv, _pooled_tex = write_measured_tasks_pooled_summary(
    _pooled_summary,
    models=_smiles_models,
)
print(f'\nWrote pooled table (not in current supplement): {_pooled_csv}')
print(f'Wrote {_pooled_tex}')
display(_pooled_summary.round(3))

_strat_summary = build_measured_tasks_stratified_summary(
    panel_scores=None,
    models=_smiles_models,
    t1_cv_scheme='predefined_fold',
)
_strat_csv, _strat_tex = write_measured_tasks_stratified_summary(
    _strat_summary,
    models=_smiles_models,
    wrap_landscape=False,
)
print(f'Wrote stratified supplement table: {_strat_csv} ({len(_strat_summary)} rows)')
for p in _strat_tex:
    print(f'Wrote {p}')
print(
    'Included in figures/submission_figures/Supplementary_Data.tex section 3 '
    '(stratified SMILES panels only; pooled / LFC-vs-Pre are not in the supplement).'
)
